# Amazon — Baseline Linear Regression
6 engineered features · Default sklearn · Saves to `baseline_LR/`

## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load data
STEP 3 — Features & scale
STEP 4 — Train & evaluate
STEP 5 — Feature importance
STEP 6 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os,json,time,warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.linear_model  import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error,mean_absolute_error

In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\data"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\baseline_LR"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print(f"Out dir: {OUT_DIR}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


---
## Step 3 — Features & Scale

In [ ]:
FEATURE_COLS = [
    'user_avg_rating', 'user_rating_count',
    'product_avg_rating', 'product_rating_count', 'product_rating_std',
    'days_since_review'
]
TARGET = 'ratings'
print(f"Features : {len(FEATURE_COLS)}  Target : {TARGET}")


In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURE_COLS])
X_val   = scaler.transform(df_val[FEATURE_COLS])
X_test  = scaler.transform(df_test[FEATURE_COLS])
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values
print(f"X_train : {X_train.shape}")


---
## Step 4 — Train & Evaluate
Plain LinearRegression — no regularisation. Absolute floor for comparison.

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4)
    mae  = round(float(mean_absolute_error(y_true, y_pred)), 4)
    return rmse, mae


In [ ]:
t0=time.perf_counter()
model=LinearRegression()
model.fit(X_train,y_train)
train_time=time.perf_counter()-t0

tr_rmse,tr_mae   = get_metrics(y_train,model.predict(X_train))
val_rmse,val_mae = get_metrics(y_val,  model.predict(X_val))
te_rmse,te_mae   = get_metrics(y_test, model.predict(X_test))

print('='*50)
print('Linear Regression — Baseline Results')
print('='*50)
print(f'  Train      RMSE: {tr_rmse}   MAE: {tr_mae}')
print(f'  Validation RMSE: {val_rmse}   MAE: {val_mae}')
print(f'  Test       RMSE: {te_rmse}   MAE: {te_mae}')
print(f'  Train time : {train_time:.2f}s')
print(f'  Train-Test gap : {round(te_rmse-tr_rmse,4)}')

---
## Step 5 — Feature Importance
Absolute coefficient value = strength of linear effect.

In [ ]:
coefs=model.coef_.ravel()
imp_df=(pd.DataFrame({'feature':FEATURE_COLS,'coef':coefs})
          .assign(abs_coef=lambda x:x['coef'].abs())
          .sort_values('abs_coef',ascending=False))
print(imp_df[['feature','coef']].to_string(index=False))

---
## Step 6 — Save Results

In [ ]:
result = {
    'model': 'LinearRegression (Baseline)',
    'train_rmse': tr_rmse, 'val_rmse': val_rmse, 'test_rmse': te_rmse,
    'train_mae':  tr_mae,  'val_mae':  val_mae,  'test_mae':  te_mae,
    'train_time_s': round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'lr_results.csv'), index=False)
with open(os.path.join(OUT_DIR, 'lr_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → {OUT_DIR}")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
